# Docker& Containerization

**Module 12 · Lesson 12.5**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Why Containerize: Kill “Works On My Machine”
- Image = Layers, and the Build Cache
- Base Image and Build Context
- Multi-Stage: Build Heavy, Ship Light
- Reproducible and Fast Dependencies with uv
- Run It Safely: Non-Root, PID 1, Healthcheck

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install numpy python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


## The most expensive sentence in software

> 💡 **Info**
>
> It worked on your laptop. The demo was flawless. Then you deployed, and it crashed on startup: a different Python version, a numpy your prod box never installed, a system library that was there in dev and missing in prod. **“Works on my machine” is the single most expensive sentence in software** - it turns a five-minute change into a two-hour incident. A **container** ends it. You freeze the exact filesystem, dependencies, runtime, and config into an **image**, and that image runs **byte-for-byte identically** on your laptop, in CI, and in production. This lesson builds the whole craft: the layer-and-cache model that makes builds fast, the base-image and multi-stage choices that make images small, the dependency discipline that makes them reproducible, the PID-1 and non-root details that make them safe, and the build → scan → push → run pipeline that ships them. Every piece runs with no Docker daemon and no API key.

### 🎯 What you will be able to do after this lesson

- **Build** a production Dockerfile - slim base, correct layer order, a multi-stage build, pinned + uv-installed deps, non-root, healthcheck.

- **Compare** base images (full / slim / alpine / distroless) on size, compatibility, CVEs, and cold start, and single-stage vs multi-stage.

- **Operate** the build cache (deps-before-code, BuildKit cache mounts) and container signals (exec-form CMD, SIGTERM graceful shutdown).

- **Evaluate** an image for production: reproducible, small, low-CVE, non-root, and healthcheck-ready.

> 📦 **Info**
>
> ✅ Before you startYou already deployed a container in **10.6** (a remote MCP server on Google Cloud Run) - that deploy hid the container work, and this lesson opens the box. It reuses **12.4**’s idea of a content-keyed cache populated on a miss (the Docker build cache is exactly that), and it builds the container that **12.1**’s reference architecture drew as one box per service. CI/CD for these images is **Lesson 12.6**; scaling and Kubernetes are **Lesson 12.7**.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 📦 **Analogy**
>
> A container image is a **meal-kit box**. A recipe alone is “works on my machine” - it assumes your kitchen already has the right flour, the right oven, the right pans, and when a friend tries it with different ingredients the dish comes out wrong. A meal kit ships the *exact* pre-measured ingredients and instructions in one sealed box, so the dish comes out identical in any kitchen. An image is that sealed box for your app: the exact runtime, libraries, and code, so it runs the same on your laptop, in CI, and in production. **Where it breaks down:** a meal kit is inert until you cook it - an image is inert until you *run* it, and the running instance is called a container.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“A container is a lightweight VM.”** It is not - it shares the host kernel and packages a filesystem plus a process. That is why it is small and fast, and also why the base image and running as non-root matter so much.
> - **“Smaller is always better.”** Alpine is tiny but its musl C library forces AI wheels to compile from source; for a Python AI app, the Debian-based **slim** image is the right default.

> 💡 **Info**
>
> ⚠️ Anti-patternTwo habits that quietly cost you. **Copying code before dependencies** (`COPY . .` then install) so every one-line code edit reinstalls every package from scratch. And the **shell form** `CMD python main.py`, which puts `/bin/sh` at PID 1 - it does not forward SIGTERM, so on a rolling deploy your app is never told to stop and its in-flight requests are killed instead of drained. Put deps before code, and use the exec form `CMD ["python","main.py"]`.

---

## 🎯 Concept 1: Why Containerize: Kill “Works On My Machine”

### Why Containerize: Kill “Works On My Machine”

A container freezes the filesystem, runtime, and dependencies into an image, so the same bytes run identically on your laptop, in CI, and in production.

Start with the problem the whole lesson solves. Your code does not run in a vacuum - it depends on a specific Python version, specific library versions, and specific system libraries. On your laptop those are whatever you happen to have installed; on the prod server they are something else. When they differ, the app crashes in a way that is maddening to debug because *the code is fine*. A **container image** removes the variable: it is a frozen snapshot of the entire environment - filesystem, interpreter, pinned dependencies, and config - bundled into one artifact. You build the image once, and that *same* image runs everywhere. The running instance of an image is a **container**. The block shows the failure and the fix, keyless.

> 🍽️ **Analogy**
>
> It is the **meal-kit box** from a moment ago, made concrete. A recipe says “add flour” and trusts your pantry; a meal kit ships the exact 200 grams of a specific flour. Ship the recipe (the code alone) and every kitchen produces a slightly different dish; ship the kit (the image) and every kitchen produces the identical dish. Containerizing is packing the kit.

The same code runs on a dev laptop (Python 3.13, numpy 2.2) and a prod box (Python 3.12, numpy 2.0). What happens without a container?

**📝 `01_reproducibility.py`** - *runnable*

In [ ]:
# Same code, different machines. Only a FROZEN image runs identically everywhere.
def build_env(python, os_name, numpy):
    return {"python": python, "os": os_name, "numpy": numpy}

def run_app(env):
    # The app was written against numpy 2.2 (it calls a 2.2-only API).
    py_minor = int(env["python"].split(".")[1])
    nx = tuple(int(p) for p in env["numpy"].split(".")[:2])
    if py_minor < 12:
        return "CRASH: needs Python 3.12+, found " + env["python"]
    if nx < (2, 2):
        return "CRASH: app uses a numpy 2.2 API, found " + env["numpy"]
    return "OK: served a request on " + env["os"]

dev  = build_env("3.13.1", "macOS-arm64",   "2.2.2")
prod = build_env("3.12.3", "linux-x86_64",  "2.0.1")   # a different box, drifted deps

print("Without a container - the SAME code on two machines:")
print("  dev  ", run_app(dev))
print("  prod ", run_app(prod))

# Containerize: freeze python + os + deps into ONE image, then run that image anywhere.
IMAGE = build_env("3.13.1", "linux (image)", "2.2.2")  # the image IS the environment
print("With a container - the SAME image on two machines:")
print("  dev-host  ", run_app(IMAGE))
print("  prod-host ", run_app(IMAGE))
print()
print("An image freezes filesystem + runtime + deps, so it runs the same everywhere.")

# Output:
# Without a container - the SAME code on two machines:
#   dev   OK: served a request on macOS-arm64
#   prod  CRASH: app uses a numpy 2.2 API, found 2.0.1
# With a container - the SAME image on two machines:
#   dev-host   OK: served a request on linux (image)
#   prod-host  OK: served a request on linux (image)
#
# An image freezes filesystem + runtime + deps, so it runs the same everywhere.

- The same `run_app` logic runs against a dev environment and a prod environment that have **drifted apart**.
- Without a container the prod box **CRASHES** - it has an older numpy than the code was written against, a bug the laptop never reproduces.
- “Containerizing” freezes one environment into an `IMAGE`; running that same image on both hosts returns **OK on both**.
- An image freezes the whole environment, so the artifact you test is byte-for-byte the artifact that runs in production.

#### 💡 What just happened

⚡ What just happened?A container image is a frozen snapshot of the filesystem, runtime, and pinned dependencies; the same image runs identically everywhere, which is what ends “works on my machine”. Tradeoff: you take on the discipline of building and versioning images, in exchange for reproducibility. An image is inert until you run it - the running instance is a container. Everything else in this lesson is about making that image small, reproducible, safe, and fast.

- Dev and prod env cards drift apart; the same code fails on prod
- Freeze both into an image and watch both hosts pass identically

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Image = Layers, and the Build Cache

### Image = Layers, and the Build Cache

A Dockerfile is an ordered list of instructions, each producing a content-hashed layer. A cache miss on one layer invalidates every layer below it, so order stable-before-volatile.

An image is not a monolith - it is a **stack of layers**, one per Dockerfile instruction, and this is the single most important mental model in the lesson. Each layer is keyed by a hash of the instruction plus its inputs plus the layer beneath it, and Docker **caches** layers between builds. On a rebuild, an unchanged layer is reused instantly; but the moment one layer’s inputs change, that layer and **every layer below it** must be rebuilt. The practical rule falls straight out of this: put **stable things first and volatile things last**. Copy `requirements.txt` and install dependencies *before* you copy your application code, because your code changes a hundred times a day and your dependencies change once a week. Get the order wrong and every one-line edit reinstalls every package. The block builds the same Dockerfile three times and shows the cache cascade, keyless.

> 📋 **Analogy**
>
> It is a **stack of transparent sheets** in an old animation. The background (the base image and the installed libraries) is drawn once on the bottom sheets and rarely changes; the character (your code) is on the top sheet and is redrawn constantly. As long as you keep the background on the bottom, you only redraw the top sheet each frame. Stack them the other way - character on the bottom - and every tiny change forces you to redraw the whole background too.

Your Dockerfile copies requirements.txt, installs deps, then copies code. You edit ONE line of code and rebuild. What is reused?

**📝 `02_layers_cache.py`** - *runnable*

In [ ]:
import hashlib
def h(*parts):
    return hashlib.md5("|".join(parts).encode()).hexdigest()[:8]

# A Dockerfile is an ORDERED list of instructions; each makes a LAYER keyed by a hash
# of (the layer below + the instruction + its inputs). A cache MISS cascades downward.
def build(instructions, prev):
    layers, cache, parent, broke = [], {}, "root", False
    for name, content in instructions:
        key = h(parent, name, content)
        if not broke and prev.get(name) == key:
            status = "CACHED"
        else:
            status, broke = "BUILD ", True   # a miss invalidates every layer below
        layers.append((status, key, name)); cache[name] = key; parent = key
    return layers, cache

def show(title, layers):
    built = sum(1 for s, _, _ in layers if s.strip() == "BUILD")
    print(title)
    for status, key, name in layers:
        print("  [" + status + "] " + key + "  " + name)
    print("  -> rebuilt " + str(built) + " layer(s)\n")

def right(code, reqs):   # deps BEFORE code (correct)
    return [("FROM  python:3.13-slim", "base"),
            ("COPY  requirements.txt", reqs),
            ("RUN   pip install -r requirements.txt", reqs),
            ("COPY  . .   (app code)", code),
            ("CMD   [python, main.py]", "cmd")]

l1, c1 = build(right("v1", "r1"), {});  show("Build 1 (cold cache): everything builds", l1)
l2, c2 = build(right("v2", "r1"), c1);  show("Build 2 (edited code, deps unchanged): deps stay cached", l2)
l3, c3 = build(right("v2", "r2"), c2);  show("Build 3 (edited requirements.txt): reinstall cascades", l3)

def wrong(code, reqs):   # code BEFORE deps (anti-pattern)
    return [("FROM  python:3.13-slim", "base"),
            ("COPY  . .   (app code)", code),
            ("COPY  requirements.txt", reqs),
            ("RUN   pip install -r requirements.txt", reqs),
            ("CMD   [python, main.py]", "cmd")]

w1, wc1 = build(wrong("v1", "r1"), {})
w2, wc2 = build(wrong("v2", "r1"), wc1)  # ONLY the code changed
show("Anti-pattern (code before deps), edit code: pip reinstalls AGAIN", w2)

# Output:
# Build 1 (cold cache): everything builds
#   [BUILD ] 2bc9207b  FROM  python:3.13-slim
#   [BUILD ] 215238fa  COPY  requirements.txt
#   [BUILD ] 97beb503  RUN   pip install -r requirements.txt
#   [BUILD ] d245c64e  COPY  . .   (app code)
#   [BUILD ] 8248bd5e  CMD   [python, main.py]
#   -> rebuilt 5 layer(s)
#
# Build 2 (edited code, deps unchanged): deps stay cached
#   [CACHED] 2bc9207b  FROM  python:3.13-slim
#   [CACHED] 215238fa  COPY  requirements.txt
#   [CACHED] 97beb503  RUN   pip install -r requirements.txt
#   [BUILD ] 32c89dc0  COPY  . .   (app code)
#   [BUILD ] 88650a5a  CMD   [python, main.py]
#   -> rebuilt 2 layer(s)
#
# Build 3 (edited requirements.txt): reinstall cascades
#   [CACHED] 2bc9207b  FROM  python:3.13-slim
#   [BUILD ] 24c29eb3  COPY  requirements.txt
#   [BUILD ] 377f7251  RUN   pip install -r requirements.txt
#   [BUILD ] fd5dde5d  COPY  . .   (app code)
#   [BUILD ] feb03504  CMD   [python, main.py]
#   -> rebuilt 4 layer(s)
#
# Anti-pattern (code before deps), edit code: pip reinstalls AGAIN
#   [CACHED] 2bc9207b  FROM  python:3.13-slim
#   [BUILD ] 4e981a45  COPY  . .   (app code)
#   [BUILD ] 89604e23  COPY  requirements.txt
#   [BUILD ] 41ff6a67  RUN   pip install -r requirements.txt
#   [BUILD ] 2afcfa54  CMD   [python, main.py]
#   -> rebuilt 4 layer(s)

- Each instruction is a **layer** with a short content hash; **Build 1** is a cold cache, so all five build.
- **Build 2** edits only the code: the base, the requirements copy, and the install stay **CACHED** - only the code layer and the CMD below it rebuild.
- **Build 3** edits requirements.txt: that layer misses, and the miss **cascades** down through the install and everything below it.
- The anti-pattern (code before deps) makes a one-line code edit **reinstall every package** - the same rebuild you were trying to avoid.

#### 💡 What just happened

⚡ What just happened?An image is an ordered stack of content-hashed layers, and a cache miss on any layer invalidates every layer below it. Tradeoff / the rule that falls out: order stable-before-volatile - copy and install dependencies before copying code - so a code edit rebuilds only the code layer, not the whole dependency install. This one ordering decision is the difference between a two-second rebuild and a ninety-second one.

- A stack of hashed layers; toggle edit-code vs edit-deps
- Watch the cache HIT/MISS cascade and the rebuild count; flip the order for the anti-pattern

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Base Image and Build Context

### Base Image and Build Context

The base image sets your size, compatibility, and CVE floor. Slim is the right default for AI (glibc); alpine is smaller but breaks AI wheels. A .dockerignore shrinks what you even send to the daemon.

Every image starts `FROM` a base, and that one line sets your floor for size, compatibility, and security. The full `python:3.13` image carries a whole Debian userland (about a gigabyte); `python:3.13-slim` strips that to the essentials and is the right default. The tempting-looking `python:3.13-alpine` is even smaller, but it uses the **musl** C library instead of **glibc**, and most AI wheels (numpy, torch, grpcio) ship prebuilt for glibc only - on alpine they have no wheel and must **compile from source**, which is slow and fragile. For hardening, **distroless** and **Chainguard** images strip the shell and package manager for a near-zero-CVE base (a **CVE** is a publicly catalogued security vulnerability, so fewer is better). Separately, the **build context** is everything you send to the daemon; a `.dockerignore` that excludes `.git`, `.venv`, `__pycache__`, and a stray `.env` shrinks it dramatically and keeps secrets out of the image. The block compares the bases and the context, keyless.

> 🧨 **Analogy**
>
> Choosing a base image is **choosing a suitcase**. The full image is a steamer trunk - it holds everything but it is heavy to carry and slow through the airport. The slim image is a well-packed carry-on: everything you actually need, nothing you do not. Alpine is an ultralight bag that saves weight but has no standard pockets, so half your gear (the AI wheels) will not fit without repacking. And the `.dockerignore` is refusing to pack the junk drawer - the old receipts and spare keys (`.git`, `.venv`, your `.env`) that should never leave the house.

You switch your base image from python:3.13-slim to python:3.13-alpine to save space. What is the likely surprise for an AI app?

**📝 `03_base_and_context.py`** - *runnable*

In [ ]:
# Base-image sizes (MB, uncompressed, illustrative 2026 figures).
BASE = {
    "python:3.13 (full)":   {"size": 1020, "libc": "glibc", "cve": 120, "note": "everything, huge"},
    "python:3.13-slim":     {"size": 150,  "libc": "glibc", "cve": 25,  "note": "glibc, AI wheels just work"},
    "python:3.13-alpine":   {"size": 50,   "libc": "musl",  "cve": 8,   "note": "musl -> compiles numpy from source"},
    "distroless/python3":   {"size": 50,   "libc": "glibc", "cve": 2,   "note": "no shell, minimal attack surface"},
}
VENV, CODE, SPEED = 180, 5, 50   # deps MB, code MB, pull MB/s

print("Base image -> final image size, CVEs, and cold-start pull time:")
print("  {:<22}{:>7}{:>6}{:>9}   {}".format("base", "img MB", "CVEs", "pull", "note"))
for name, b in BASE.items():
    total = b["size"] + VENV + CODE
    print("  {:<22}{:>7}{:>6}{:>9}   {}".format(name, total, b["cve"], format(total / SPEED, ".1f") + "s", b["note"]))
print()
print("For AI: slim is the sweet spot - glibc means numpy/torch install as prebuilt wheels;")
print("alpine's musl forces a from-source compile; distroless/Chainguard cut CVEs further.")
print()

# .dockerignore: what gets sent to the daemon as the build CONTEXT.
context = {"app code": 5, ".git/": 120, ".venv/": 180, "__pycache__/": 15, "tests/": 20, ".env (secret!)": 1}
full_ctx, kept_ctx = sum(context.values()), 5
print(".dockerignore - the build context sent to the daemon:")
print("  without .dockerignore: {} MB  ({})".format(full_ctx, ", ".join(context)))
print("  with    .dockerignore: {} MB  (app code only; .git/.venv/__pycache__/.env excluded)".format(kept_ctx))
print("  -> {}x smaller context, faster uploads, and no .env leaking into a layer".format(round(full_ctx / kept_ctx)))

# Output:
# Base image -> final image size, CVEs, and cold-start pull time:
#   base                   img MB  CVEs     pull   note
#   python:3.13 (full)       1205   120    24.1s   everything, huge
#   python:3.13-slim          335    25     6.7s   glibc, AI wheels just work
#   python:3.13-alpine        235     8     4.7s   musl -> compiles numpy from source
#   distroless/python3        235     2     4.7s   no shell, minimal attack surface
#
# For AI: slim is the sweet spot - glibc means numpy/torch install as prebuilt wheels;
# alpine's musl forces a from-source compile; distroless/Chainguard cut CVEs further.
#
# .dockerignore - the build context sent to the daemon:
#   without .dockerignore: 341 MB  (app code, .git/, .venv/, __pycache__/, tests/, .env (secret!))
#   with    .dockerignore: 5 MB  (app code only; .git/.venv/__pycache__/.env excluded)
#   -> 68x smaller context, faster uploads, and no .env leaking into a layer

- The base sets the floor: **full** is huge with many CVEs; **slim** keeps glibc so AI wheels just work; **alpine** is small but musl forces a from-source compile; **distroless** cuts CVEs hardest.
- Image size drives the **cold-start pull time** - a bigger base is a slower first request on a scale-from-zero platform.
- The **build context** is what you send to the daemon; without a `.dockerignore` your `.git`, `.venv`, and even a `.env` secret ship along.
- A `.dockerignore` that keeps only the app code shrinks the context dramatically and keeps secrets out of a layer.

#### 💡 What just happened

⚡ What just happened?The base image sets your size, compatibility, and CVE floor; slim (Debian, glibc) is the right default for AI because alpine’s musl breaks prebuilt wheels, and distroless/Chainguard harden further. Tradeoff: smaller is not automatically better - you trade compatibility and debuggability for size. The .dockerignore shrinks the build context and keeps .git, .venv, and .env out of the image.

- Pick full/slim/alpine/distroless: size, CVE, and pull-time bars
- An alpine 'compiles numpy from source' warning; toggle .dockerignore to shrink the context

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Multi-Stage: Build Heavy, Ship Light

### Multi-Stage: Build Heavy, Ship Light

A builder stage installs compilers and builds the dependencies; the runtime stage copies only the finished virtual environment. Compilers and caches never ship to production.

To install AI dependencies you often need a compiler - gcc and make to build numpy or grpcio from source when a wheel is missing. But you do **not** need that compiler to *run* the app, and shipping it to production just bloats the image and widens the attack surface. A **multi-stage build** solves this cleanly: a first `builder` stage installs `build-essential` and compiles everything into a virtual environment, and a second `runtime` stage starts from a fresh slim base and copies **only the finished venv** (`COPY --from=builder ...`). The builder - with its compilers, headers, and apt caches - is discarded. The final image is dramatically smaller and carries zero build tools. The block computes both sizes, keyless.

> 🛠️ **Analogy**
>
> It is a **workshop versus the display case**. A furniture maker’s workshop is full of saws, clamps, sawdust, and offcuts - everything needed to *build* the chair. But the showroom displays only the finished chair; you would never ship the whole workshop to the customer’s living room. The builder stage is the workshop (compilers, headers, caches); the runtime stage is the display case (just the finished venv and your code). Build heavy, ship light.

A single-stage image is 580 MB with gcc and make inside. You convert it to multi-stage. What happens to the final image?

**📝 `04_multistage.py`** - *runnable*

In [ ]:
BASE, BUILD_TOOLS, VENV, APT_CACHE, CODE = 150, 200, 180, 45, 5   # MB (illustrative)

single = BASE + BUILD_TOOLS + VENV + APT_CACHE + CODE
print("SINGLE-STAGE (build tools ship to production):")
print("  base {} + build-tools {} + venv {} + apt-cache {} + code {} = {} MB".format(
    BASE, BUILD_TOOLS, VENV, APT_CACHE, CODE, single))
print("  build tools in the final image: gcc, make (2)")
print()

# Multi-stage: the builder compiles; the runtime copies ONLY the venv.
builder = BASE + BUILD_TOOLS + VENV     # exists only during the build, then discarded
runtime = BASE + VENV + CODE
print("MULTI-STAGE (builder compiles, runtime ships only the venv):")
print("  builder (discarded): base {} + build-tools {} + venv {} = {} MB".format(BASE, BUILD_TOOLS, VENV, builder))
print("  runtime  (shipped):  base {} + venv {} + code {} = {} MB".format(BASE, VENV, CODE, runtime))
print("  build tools in the final image: none (0)")
print()
print("  saved {} MB ({}% smaller), and 0 compilers in production = smaller attack surface".format(
    single - runtime, round((single - runtime) / single * 100)))

# Output:
# SINGLE-STAGE (build tools ship to production):
#   base 150 + build-tools 200 + venv 180 + apt-cache 45 + code 5 = 580 MB
#   build tools in the final image: gcc, make (2)
#
# MULTI-STAGE (builder compiles, runtime ships only the venv):
#   builder (discarded): base 150 + build-tools 200 + venv 180 = 530 MB
#   runtime  (shipped):  base 150 + venv 180 + code 5 = 335 MB
#   build tools in the final image: none (0)
#
#   saved 245 MB (42% smaller), and 0 compilers in production = smaller attack surface

- The **single-stage** image carries the base, the build tools, the venv, an apt cache, and the code - and gcc/make ship to production.
- The **multi-stage** build runs the same compile in a `builder` that is then **discarded**.
- The `runtime` stage copies **only the venv** onto a fresh slim base - the same app, no compilers.
- The result is a large size cut and **zero build tools in production**, which also shrinks the attack surface.

#### 💡 What just happened

⚡ What just happened?A multi-stage build compiles dependencies in a throwaway builder stage and copies only the finished virtual environment into a clean runtime stage. Tradeoff: a slightly more complex Dockerfile, in exchange for a much smaller image with no compilers or apt caches in production. Smaller image plus smaller attack surface plus faster cold starts - build heavy, ship light.

- A builder box (base + gcc + venv) and a runtime box that copies only the venv
- A size bar drops 580 to 335 and a build-tools-shipped counter 2 to 0

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Reproducible and Fast Dependencies with uv

### Reproducible and Fast Dependencies with uv

Pin exact versions and lock the tree so builds are reproducible; use uv and a BuildKit cache mount so installs go from tens of seconds to under a second.

Two dependency problems, one step. First, **reproducibility**: an unpinned `fastapi` means “whatever is latest at build time”, which resolves to one version today and a different one next week - the exact drift that causes “works on my machine”. The fix is to **pin every version** and commit a **lock file** so the whole dependency tree is frozen. Second, **speed**: `pip` is slow, and in CI you build images many times a day. **uv** (a Rust-based installer from Astral) resolves and installs an order of magnitude faster, and a **BuildKit cache mount** (BuildKit is Docker’s default build engine; `--mount=type=cache`) persists the download cache across builds so the second build barely touches the network. Setting `UV_COMPILE_BYTECODE=1` precompiles the `.pyc` files so the container also *starts* faster. The block shows drift and the install-time difference, keyless.

> 🧊 **Analogy**
>
> Pinning is a **recipe with exact grams**. “Some flour, a bit of sugar” gives a different cake every time and in every kitchen; “200 g flour, 50 g sugar” gives the identical cake, forever. An unpinned requirement is “some flour”; a pinned one with a lock file is the exact gram list. And uv with a cache mount is having the ingredients pre-measured and waiting on the counter - you skip the trip to the store on every bake.

Your requirements list `fastapi` with no version. You build today and again next month. What is the danger?

**📝 `05_deps.py`** - *runnable*

In [ ]:
# What "latest" resolves to on two different days (illustrative index).
INDEX = {
    "day-1": {"fastapi": "0.115.6", "uvicorn": "0.34.0", "pydantic": "2.10.5"},
    "day-2": {"fastapi": "0.118.0", "uvicorn": "0.37.1", "pydantic": "2.11.9"},
}
def resolve(spec, day):
    name, _, pin = spec.partition("==")
    return pin if pin else INDEX[day][name]

print("Reproducibility - resolve the same requirement on two days:")
for spec in ["fastapi", "fastapi==0.115.6"]:
    d1, d2 = resolve(spec, "day-1"), resolve(spec, "day-2")
    tag = "SAME (reproducible)" if d1 == d2 else "DRIFTED (works on my machine!)"
    print("  {:<20} day-1={:<9} day-2={:<9} -> {}".format(spec, d1, d2, tag))
print()

# Install speed: 8 packages, cold pip vs uv vs a BuildKit cache mount.
N, pip_each, uv_each, cache_each = 8, 4.0, 0.40, 0.05
print("Install speed - {} packages:".format(N))
print("  pip (cold)                 {:>6.1f}s".format(N * pip_each))
print("  uv  (cold, Rust resolver)  {:>6.1f}s   ({}x faster than pip)".format(N * uv_each, round(pip_each / uv_each)))
print("  uv  + BuildKit cache mount {:>6.1f}s   (wheels already downloaded)".format(N * cache_each))
print()
print("Pin every version and commit a lock file for reproducibility; use uv + a cache mount for speed.")

# Output:
# Reproducibility - resolve the same requirement on two days:
#   fastapi              day-1=0.115.6   day-2=0.118.0   -> DRIFTED (works on my machine!)
#   fastapi==0.115.6     day-1=0.115.6   day-2=0.115.6   -> SAME (reproducible)
#
# Install speed - 8 packages:
#   pip (cold)                   32.0s
#   uv  (cold, Rust resolver)     3.2s   (10x faster than pip)
#   uv  + BuildKit cache mount    0.4s   (wheels already downloaded)
#
# Pin every version and commit a lock file for reproducibility; use uv + a cache mount for speed.

- An **unpinned** requirement resolves to different versions on two different days - a build that is not reproducible.
- A **pinned** requirement (`fastapi==0.115.6`) resolves to the same version every time, on every machine.
- On install speed, **uv** is roughly an order of magnitude faster than pip on a cold install.
- A **BuildKit cache mount** reuses already-downloaded wheels, so a rebuild’s install drops to a fraction of a second.

#### 💡 What just happened

⚡ What just happened?Pin every version and commit a lock file so the dependency tree is frozen and reproducible; use uv plus a BuildKit cache mount so installs go from tens of seconds to a fraction of one. Tradeoff: you maintain a lock file and adopt a newer tool, in exchange for reproducible builds and much faster CI. UV_COMPILE_BYTECODE also precompiles .pyc so the container starts faster.

- Unpinned fastapi resolves to different versions on day 1 vs day 2; pinned matches
- An install-time race: pip vs uv vs a BuildKit cache mount

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Run It Safely: Non-Root, PID 1, Healthcheck

### Run It Safely: Non-Root, PID 1, Healthcheck

Run as a non-root user, use the exec-form CMD so your app is PID 1 and receives SIGTERM for graceful shutdown, and add a healthcheck so an unhealthy instance is restarted.

A small, reproducible image can still be a production hazard if it runs wrong. Three details matter. **Non-root:** by default a container runs as `root` (uid 0), so any escape has full host access - create a user and `USER appuser` to shrink the blast radius. **Signals and PID 1:** when the platform rolls out a new version it sends **SIGTERM** to process 1 and waits about ten seconds before a hard **SIGKILL**. If you use the shell form `CMD python main.py`, then `/bin/sh` is PID 1 and it does *not* forward SIGTERM - your app is never told to stop, so its in-flight requests are killed mid-flight. Use the **exec form** `CMD ["python","main.py"]` so your app is PID 1 and can catch SIGTERM to drain requests gracefully (uvicorn does this for you). **Healthcheck:** a `HEALTHCHECK` (or Cloud Run probe) lets the platform restart an instance that has gone unhealthy. Two more safety rules that do not fit in the block but matter just as much: **never bake a secret into the image** - an `ENV ANTHROPIC_API_KEY=...`, an `ARG`, or a copied `.env` becomes a permanent part of the immutable layer history that anyone who pulls the image can read, so inject secrets at *runtime* instead (for example Cloud Run `--set-secrets`). And a container is **ephemeral**: its writable layer is discarded on restart, so durable state must live outside it - a database, an object store, or a mounted volume - never on the container’s local disk. The block models non-root, signals, and the healthcheck, keyless.

> 🛡️ **Analogy**
>
> Non-root is **giving the night guard a key to one office, not the master key to the whole building** - if the guard is compromised, the damage is contained. And graceful shutdown is a **fire drill**: a good building gives people time to file out calmly (drain in-flight requests) before the doors lock; the shell-form mistake is locking the doors the instant the alarm sounds, with people still inside.

On a rolling deploy the platform sends SIGTERM to PID 1. Your Dockerfile ends with the shell form `CMD python main.py`. What happens to in-flight requests?

**📝 `06_run_safety.py`** - *runnable*

In [ ]:
# 1) Blast radius: root vs non-root if the container is breached.
def blast(uid):
    return "FULL host access (write anywhere, install, escalate)" if uid == 0 else "limited to the app's own files"
print("Non-root - the blast radius if the container is breached:")
print("  USER root    (uid 0)    -> " + blast(0))
print("  USER appuser (uid 1000) -> " + blast(1000))
print()

# 2) Graceful shutdown. Docker sends SIGTERM to pid 1, waits 10s, then SIGKILL.
def deploy(cmd_form):
    inflight = 3
    if cmd_form == "exec":   # CMD ["python","main.py"] -> python IS pid 1, handles SIGTERM
        pid1, handled = "python (your app)", True
    else:                    # CMD python main.py -> /bin/sh is pid 1, does not forward SIGTERM
        pid1, handled = "/bin/sh", False
    outcome = "drained gracefully" if handled else "KILLED at the 10s SIGKILL - requests dropped"
    print("  CMD {:<6} pid1={:<18} SIGTERM handled={!s:<5} -> {} in-flight {}".format(
        cmd_form, pid1, handled, inflight, outcome))

print("Graceful shutdown - a rolling deploy sends SIGTERM to pid 1:")
deploy("shell")
deploy("exec")
print()

# 3) Healthcheck: three consecutive failures restart the instance.
print("Healthcheck - three strikes restart the instance:")
fails = 0
for i, code in enumerate([200, 200, 500, 500, 500], 1):
    fails = 0 if code == 200 else fails + 1
    action = "RESTART (3 strikes)" if fails >= 3 else "ok"
    print("  probe {} -> HTTP {}  fails={}  {}".format(i, code, fails, action))

# Output:
# Non-root - the blast radius if the container is breached:
#   USER root    (uid 0)    -> FULL host access (write anywhere, install, escalate)
#   USER appuser (uid 1000) -> limited to the app's own files
#
# Graceful shutdown - a rolling deploy sends SIGTERM to pid 1:
#   CMD shell  pid1=/bin/sh            SIGTERM handled=False -> 3 in-flight KILLED at the 10s SIGKILL - requests dropped
#   CMD exec   pid1=python (your app)  SIGTERM handled=True  -> 3 in-flight drained gracefully
#
# Healthcheck - three strikes restart the instance:
#   probe 1 -> HTTP 200  fails=0  ok
#   probe 2 -> HTTP 200  fails=0  ok
#   probe 3 -> HTTP 500  fails=1  ok
#   probe 4 -> HTTP 500  fails=2  ok
#   probe 5 -> HTTP 500  fails=3  RESTART (3 strikes)

- Running as **root** gives a breach full host access; a **non-root** user limits the blast radius to the app’s own files.
- With the **shell form**, `/bin/sh` is PID 1 and swallows SIGTERM, so the three in-flight requests are **killed** at the hard SIGKILL.
- With the **exec form**, your app is PID 1, catches SIGTERM, and **drains** the in-flight requests gracefully.
- The **healthcheck** restarts an instance after three consecutive failed probes, so a wedged container does not keep serving errors.

#### 💡 What just happened

⚡ What just happened?Run as a non-root user to shrink the blast radius; use the exec-form CMD so your app is PID 1 and can catch SIGTERM to drain in-flight requests on a rolling deploy; add a healthcheck so the platform restarts an unhealthy instance. Tradeoff: a few extra lines in the Dockerfile, in exchange for safe restarts and no dropped requests. The shell-form CMD is the single most common cause of dropped requests on deploy.

- Two containers, exec-form vs shell-form CMD; press deploy to send SIGTERM
- Exec-form drains 3 in-flight requests, shell-form drops them at the 10-second SIGKILL; a root vs non-root blast-radius meter

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Ship It: Build, Scan, Push, Run

### Ship It: Build, Scan, Push, Run

Build with BuildKit, scan the image for CVEs, push only the changed layers to a registry, and run it on a platform like Cloud Run. Every earlier step compounds here.

Now put it on the internet. The shipping pipeline is four steps. **Build** the image (BuildKit is the default builder and gives you the cache mounts from Step 5). **Scan** it for known vulnerabilities with **Trivy** or **Docker Scout** before it ships - a slimmer, hardened base means far fewer findings, which is where Steps 3 and 4 pay off. **Push** it to a registry, which uploads **only the layers that changed** (content-addressed dedup) - so keeping the volatile code layer last, from Step 2, means a redeploy pushes a few megabytes instead of the whole image. **Run** it on a platform - `gcloud run deploy --source` builds via Cloud Build and runs it on Cloud Run, injecting the `$PORT` your app must listen on. And the whole chain compounds: a smaller image scans cleaner, pushes faster, and cold-starts quicker. The block runs a fat pipeline against a slim one, keyless.

> ✈️ **Analogy**
>
> It is **getting on a flight**. You go through security (the scan - a fat bag full of junk gets flagged more), you check only what changed since last trip (the push - you do not re-check bags already at the gate), and you board (the run). A traveller who packed light (slim, multi-stage) breezes through every checkpoint; the one hauling a steamer trunk is slow at security, slow to load, and last off the plane.

You redeploy after a one-line code change. Your 335 MB image’s deps layer is unchanged. How much gets pushed to the registry?

**📝 `07_ship_pipeline.py`** - *runnable*

In [ ]:
# Two pipelines, end to end: build -> scan -> push -> run.
def pipeline(name, img_mb, cve, push_mb, push_note):
    cold = img_mb / 100.0    # cold start roughly scales with image size (illustrative)
    print("  " + name)
    print("    build -> image {} MB".format(img_mb))
    print("    scan  -> {} known CVEs".format(cve))
    print("    push  -> {} MB uploaded ({})".format(push_mb, push_note))
    print("    run   -> ~{:.1f}s cold start".format(cold))
    return {"mb": img_mb, "cve": cve, "push": push_mb, "cold": cold}

print("Pipeline A - fat, single-stage, unscanned base:")
a = pipeline("A", 1205, 120, 1205, "first push - every layer")     # full base, all layers
print()
print("Pipeline B - slim, multi-stage, scanned:")
b = pipeline("B", 335, 25, 5, "only the changed code layer")       # deps layer already in the registry
print()
print("Slim multi-stage vs fat single-stage:")
print("  image   {} MB -> {} MB   ({}% smaller)".format(a["mb"], b["mb"], round((a["mb"] - b["mb"]) / a["mb"] * 100)))
print("  CVEs    {} -> {}".format(a["cve"], b["cve"]))
print("  push    {} MB -> {} MB   (layer dedup)".format(a["push"], b["push"]))
print("  cold    {:.1f}s -> {:.1f}s".format(a["cold"], b["cold"]))
print("  Every earlier step compounds: base image -> size -> CVEs -> push time -> cold start.")

# Output:
# Pipeline A - fat, single-stage, unscanned base:
#   A
#     build -> image 1205 MB
#     scan  -> 120 known CVEs
#     push  -> 1205 MB uploaded (first push - every layer)
#     run   -> ~12.1s cold start
#
# Pipeline B - slim, multi-stage, scanned:
#   B
#     build -> image 335 MB
#     scan  -> 25 known CVEs
#     push  -> 5 MB uploaded (only the changed code layer)
#     run   -> ~3.4s cold start
#
# Slim multi-stage vs fat single-stage:
#   image   1205 MB -> 335 MB   (72% smaller)
#   CVEs    120 -> 25
#   push    1205 MB -> 5 MB   (layer dedup)
#   cold    12.1s -> 3.4s
#   Every earlier step compounds: base image -> size -> CVEs -> push time -> cold start.

- **Pipeline A** (fat, single-stage, unscanned) ships a huge image with many latent CVEs, pushes every layer, and cold-starts slowly.
- **Pipeline B** (slim, multi-stage, scanned) is a fraction of the size, has far fewer CVEs, and pushes **only the changed code layer**.
- Cold start roughly tracks image size, so the slim image is also the faster first request on a scale-from-zero platform.
- Every earlier step **compounds** here: the base image drives size, size drives push time and cold start, and the base drives the CVE count.

**📝 `Dockerfile + deploy`** - *illustrative*

In [ ]:
%%writefile Dockerfile
# THE PRODUCTION DOCKERFILE - an illustrative minimal example (multi-stage + uv + non-root + healthcheck).
# syntax=docker/dockerfile:1

# ---- STAGE 1: builder (compiles deps with uv, then is discarded) ----
FROM python:3.13-slim AS builder
COPY --from=ghcr.io/astral-sh/uv:0.8.4 /uv /uvx /usr/local/bin/   # pin uv too - Step 5: pin every input
ENV UV_COMPILE_BYTECODE=1 UV_LINK_MODE=copy      # precompile .pyc; copy (not hardlink) across the cache mount
WORKDIR /app
# deps FIRST (a cached layer), with a BuildKit cache mount for uv's download cache
COPY pyproject.toml uv.lock ./
RUN --mount=type=cache,target=/root/.cache/uv uv sync --frozen --no-install-project --no-editable
COPY . .
RUN --mount=type=cache,target=/root/.cache/uv uv sync --frozen --no-editable

# ---- STAGE 2: runtime (ships only the venv + code, no build tools) ----
FROM python:3.13-slim AS runtime
RUN useradd --create-home appuser
COPY --from=builder --chown=appuser /app/.venv /app/.venv
COPY --from=builder --chown=appuser /app /app
ENV PATH="/app/.venv/bin:$PATH" PYTHONUNBUFFERED=1
WORKDIR /app
USER appuser                                     # non-root: shrink the blast radius
EXPOSE 8080
HEALTHCHECK --interval=30s --timeout=5s --retries=3 \
  CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8080/health')" || exit 1
# exec form: uvicorn is pid 1 and receives SIGTERM -> drains in-flight requests
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8080"]

# --- Build, scan, and ship (shell) ---
#   docker build -t netsetos-ai:$(git rev-parse --short HEAD) .    # tag with the git SHA (immutable), never :latest
#   docker run --rm -p 8080:8080 netsetos-ai:...     # smoke-test locally first, then: curl localhost:8080/health
#   trivy image netsetos-ai:...            # or: docker scout cves netsetos-ai:...
#   gcloud run deploy netsetos-ai --source . --region asia-south1 --port 8080 --set-secrets ANTHROPIC_API_KEY=...
# Output: (illustrative - needs Docker + a uv.lock + gcloud; sizes and CVE counts vary by machine.)

- The **builder** stage copies the `uv` binary, uses a **cache mount** to install deps first (a cached layer), then installs the project - Steps 2, 4, and 5 in one file.
- The **runtime** stage copies only the venv onto a fresh slim base, runs as `appuser`, and declares a `HEALTHCHECK` - Steps 3 and 6.
- The **exec-form** `CMD` makes uvicorn PID 1 so it receives SIGTERM and drains requests on a rolling deploy.
- The shell block is the ship step: `docker build`, then `trivy`/`docker scout` to scan, then `gcloud run deploy --source` to run it on Cloud Run.

#### 💡 What just happened

⚡ What just happened?The ship pipeline is build (BuildKit) -> scan (Trivy / Docker Scout) -> push (only changed layers) -> run (Cloud Run injects $PORT). Tradeoff / the whole lesson: a few more pipeline steps, but every earlier choice compounds - a slim, multi-stage, layer-ordered image scans cleaner, pushes in megabytes, and cold-starts in seconds. CI/CD that runs this on every push is Lesson 12.6.

- One image flows build -> scan -> push -> run; toggle fat single-stage vs slim multi-stage
- Compounding readouts: size, CVEs, pushed MB, and cold start

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> A production container is a **layered** discipline, and the Dockerfile reads top to bottom as this lesson. A container ends “works on my machine” by freezing the environment into an **image** (Step 1). That image is a stack of **content-hashed layers**, so you order deps before code to keep the build cache warm (Step 2). You choose a **slim base** for glibc compatibility and a `.dockerignore` to keep the context clean (Step 3), use a **multi-stage build** so compilers never ship (Step 4), **pin and lock** dependencies and install them fast with **uv** and a cache mount (Step 5), and run it **safely** as non-root with an exec-form CMD for graceful shutdown and a healthcheck (Step 6). Then you **build, scan, push, and run** it, where every earlier choice compounds (Step 7). Ask four questions of any image: is it **reproducible**, is it **small**, is it **safe** (non-root, low-CVE), and is it **healthy** (drains on SIGTERM, restarts on failure)?

| Base image | Approx size | Compatibility | Best for |
|---|---|---|---|
| python:3.13(full) | largest (about a gigabyte) | glibc; everything included | quick experiments, not production |
| python:3.13-slim | small (glibc) | glibc; AI wheels just work | the default for Python AI apps |
| python:3.13-alpine | smallest | musl; AI wheels compile from source | tiny pure-Python apps only |
| distroless / Chainguard | small, hardened | glibc; no shell or package manager | lowest CVE surface, harder to debug |

> 📦 **Info**
>
> ➡️ Where this goes nextYou can now build a small, reproducible, safe image. Running this pipeline automatically on every push - build, test, scan, deploy - is **continuous integration and delivery**, covered in Lesson 12.6. Scaling many containers with an orchestrator like **Kubernetes**, plus autoscaling and GPU pools, is Lesson 12.7, and monitoring the running container is Lesson 12.8. The deep Cloud Run request lifecycle and auth you met in Lesson 10.6. The primary references are the [Docker build best-practices](https://docs.docker.com/build/building/best-practices/), [multi-stage builds](https://docs.docker.com/build/building/multi-stage/), the [Astral uv Docker guide](https://docs.astral.sh/uv/guides/integration/docker/), [Google distroless](https://github.com/GoogleContainerTools/distroless), [tini](https://github.com/krallin/tini) for PID-1 signals, and [Cloud Run deploy from source](https://docs.cloud.google.com/run/docs/deploying-source-code).

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Docker& Containerization**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-12_5.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-12.5.html` - regenerate this notebook whenever the source HTML is updated.*
